In [0]:
SELECT * FROM course_training_catalog.silver_olist.customers;

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
a905baa530258422594f1b05615bd225,c80da60feddb7cf8325bd104032e314a,60140,fortaleza,CE
b8cbb2fa6090f51abb6076e2b26ca3ae,c7797b25e6175614c2c8ad82742a2ddd,7263,guarulho,SP
364f824c824823a6e6dcfce11fa7cad0,a21a8c17fb2d281bb4890748efc12200,32210,contagem,MG
a70394b12364e5ae44f4553c76076a6a,0dbf75634e9cdb8dfef760501cf983a8,63010,juazeiro do norte,CE
451cb85396e597ae3bf87cfa3fd14f9d,fa01b801b0581c39cc4140fb75004d6b,27253,volta redonda,RJ
7711f97f5a39689cb92b2a1d87ca957b,da33e0d632e94d62a0edefd1ef4cc192,25808,tre rio,RJ
166cb796fb9a2d5c5ce8bede811f3839,d72a0b0a95a97197efb0d0d20978f6cc,15091,ao jo e do rio preto,SP
8947279bb0e0ba714b6900e24de09899,11dece2a80b983d52a4250e0cfac2767,65110,ao jo e de ribamar,MA
4a4dcb4f3051d863f72ad139c4b065e0,43bbfd3cde007be301013b9425014b35,84178,ca tro,PR
6fa7a221fb6f0690ff40df40ceb7c786,0b8ed32525608a907c7331b613b5b67e,3634,ao paulo,SP


In [0]:
%sql
CREATE OR REPLACE FUNCTION course_training_catalog.silver_olist.filter_customer_region(
    state STRING
)
RETURNS BOOLEAN
RETURN state IN ('SP', 'RJ');

In [0]:
%sql
SELECT DISTINCT customer_state
FROM course_training_catalog.silver_olist.customers
ORDER BY customer_state;

customer_state
RJ
SP


In [0]:
%sql
SELECT
    customer_id,
    customer_city,
    customer_state
FROM course_training_catalog.silver_olist.customers
LIMIT 20;

customer_id,customer_city,customer_state
b8cbb2fa6090f51abb6076e2b26ca3ae,guarulho,SP
451cb85396e597ae3bf87cfa3fd14f9d,volta redonda,RJ
7711f97f5a39689cb92b2a1d87ca957b,tre rio,RJ
166cb796fb9a2d5c5ce8bede811f3839,ao jo e do rio preto,SP
6fa7a221fb6f0690ff40df40ceb7c786,ao paulo,SP
f5d9940e5da79a1324329fa16246e012,carapicuiba,SP
293468c778c7dfa812bf7ed8b766577e,rio grande da erra,SP
87e5dbec1cdacf711716cf07f8b8498b,ao paulo,SP
c19ce4ec557c4b0d86dc3fa47a542502,itapetininga,SP
3a62a8f729ab13f742c0d4f797e61278,ao jo e do campo,SP


In [0]:
%sql
SELECT DISTINCT geolocation_state
FROM course_training_catalog.silver_olist.geolocation
ORDER BY geolocation_state;

geolocation_state
AC
AL
AM
AP
BA
CE
DF
ES
GO
MA


In [0]:
%sql
SELECT DISTINCT geolocation_state
FROM course_training_catalog.silver_olist.geolocation
ORDER BY geolocation_state;

geolocation_state
RJ
SP



# ABAC Column Masking with Governed Tags

In this section, we apply column-level masking to the
order_payments table using governed tags, SQL functions,
and a centralized ABAC policy.


In [0]:
%sql
select * from course_training_catalog.silver_olist.order_payments;

order_id,payment_sequential,payment_type,payment_installments,payment_value,payment_per_installment
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,12.42
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,13.47
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,64.23
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12,48.06
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84,17.28
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09,56.85
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95,51.95


In [0]:
ALTER TABLE course_training_catalog.silver_olist.order_payments
ALTER COLUMN payment_type
SET TAGS ('payment_classification' = 'restricted');

In [0]:
%sql
CREATE OR REPLACE FUNCTION course_training_catalog.silver_olist.mask_payment_type(
    payment_method STRING
)
RETURNS STRING
RETURN 'MASKED';

In [0]:
%sql
select * from course_training_catalog.silver_olist.order_payments;

order_id,payment_sequential,payment_type,payment_installments,payment_value,payment_per_installment
b81ef226f3fe1789b1e8b2acac839d17,1,MASKED,8,99.33,12.42
a9810da82917af2d9aefd1278f1dcfa0,1,MASKED,1,24.39,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,MASKED,1,65.71,65.71
ba78997921bbcdc1373bb41e913ab953,1,MASKED,8,107.78,13.47
42fdf880ba16b47b59251dd489d4441a,1,MASKED,2,128.45,64.23
298fcdf1f73eb413e4d26d01b25bc1cd,1,MASKED,2,96.12,48.06
771ee386b001f06208a7419e4fc1bbd7,1,MASKED,1,81.16,81.16
3d7239c394a212faae122962df514ac7,1,MASKED,3,51.84,17.28
1f78449c87a54faf9e96e88ba1491fa9,1,MASKED,6,341.09,56.85
0573b5e23cbd798006520e1d5b4c6714,1,MASKED,1,51.95,51.95


In [0]:
%sql
ALTER SCHEMA course_training_catalog.silver_olist
UNSET TAGS ('data_classification');

In [0]:
SHOW EFFECTIVE POLICIES
ON TABLE course_training_catalog.silver_olist.order_payments;

Policy Name,Policy Type,Catalog,Schema,Table,Comment,on_securable_type,on_securable_fullname
mask_payment_type_abac,COLUMN_MASK,course_training_catalog,silver_olist,order_payments,Masks payment type information using governed tags.,TABLE,course_training_catalog.silver_olist.order_payments
filter_regions_abac,ROW_FILTER,course_training_catalog,silver_olist,null,Allows access to SP and RJ state records using governed tags.,SCHEMA,course_training_catalog.silver_olist


In [0]:
%sql
SELECT
    order_id,
    payment_type,
    payment_value
FROM course_training_catalog.silver_olist.order_payments
LIMIT 20;

order_id,payment_type,payment_value
b81ef226f3fe1789b1e8b2acac839d17,MASKED,99.33
a9810da82917af2d9aefd1278f1dcfa0,MASKED,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,MASKED,65.71
ba78997921bbcdc1373bb41e913ab953,MASKED,107.78
42fdf880ba16b47b59251dd489d4441a,MASKED,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,MASKED,96.12
771ee386b001f06208a7419e4fc1bbd7,MASKED,81.16
3d7239c394a212faae122962df514ac7,MASKED,51.84
1f78449c87a54faf9e96e88ba1491fa9,MASKED,341.09
0573b5e23cbd798006520e1d5b4c6714,MASKED,51.95


In [0]:
DROP POLICY mask_payment_type_abac
ON TABLE course_training_catalog.silver_olist.order_payments;

In [0]:
SHOW EFFECTIVE POLICIES
ON TABLE course_training_catalog.silver_olist.order_payments;

Policy Name,Policy Type,Catalog,Schema,Table,Comment,on_securable_type,on_securable_fullname
filter_regions_abac,ROW_FILTER,course_training_catalog,silver_olist,null,Allows access to SP and RJ state records using governed tags.,SCHEMA,course_training_catalog.silver_olist


In [0]:
%sql
SELECT
    order_id,
    payment_type,
    payment_value
FROM course_training_catalog.silver_olist.order_payments
LIMIT 20;

order_id,payment_type,payment_value
b81ef226f3fe1789b1e8b2acac839d17,credit_card,99.33
a9810da82917af2d9aefd1278f1dcfa0,credit_card,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,credit_card,65.71
ba78997921bbcdc1373bb41e913ab953,credit_card,107.78
42fdf880ba16b47b59251dd489d4441a,credit_card,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,credit_card,96.12
771ee386b001f06208a7419e4fc1bbd7,credit_card,81.16
3d7239c394a212faae122962df514ac7,credit_card,51.84
1f78449c87a54faf9e96e88ba1491fa9,credit_card,341.09
0573b5e23cbd798006520e1d5b4c6714,boleto,51.95


In [0]:
USE CATALOG course_training_catalog;

SHOW USER FUNCTIONS IN silver_olist;

function
course_training_catalog.silver_olist.filter_customer_region
course_training_catalog.silver_olist.filter_seller_state
course_training_catalog.silver_olist.mask_payment_type
course_training_catalog.silver_olist.mask_seller_zip


In [0]:
%sql
DESCRIBE FUNCTION EXTENDED
course_training_catalog.silver_olist.mask_payment_type;

function_desc
Function: course_training_catalog.silver_olist.mask_payment_type
Type: SCALAR
Input: payment_method STRING
Returns: STRING
Deterministic: true
Data Access: CONTAINS SQL
Configs: runtime.databricks.sql.functions.counter_diff.enabled=true
runtime.databricks.sql.functions.time_bucket.enabled=true
spark.databricks.ai.adaptiveBatch.initialSize=4
spark.databricks.ai.adaptiveBatch.maxRetries=100
